# Results: compare finished runs

The runs' saved files, read by `core/runner/results.py` (the Studio's Results page
reads the same): this PC's runs and those of each cluster **after Sync results**
(`runs/@<target>/`). Nothing here trains, starts Ray or connects to the cluster.

Three questions, one section each:

1. **Did it learn?** Evaluation return against environment steps.
2. **How good is the final policy?** The held-out test days, the trained policy next to
   the `zero` and `net_position` baselines, paired per day.
3. **How consistent is it?** Each configuration over its training seeds. A seed is one
   run; the days and participants of one run are not independent results.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Repository root: found from wherever the notebook runs, and put on the path.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core" / "rl").is_dir())
sys.path.insert(0, str(ROOT))

from core.runner import results

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## The runs

`find_runs` lists every run with its target; `load_runs` reads its settings and state.
Test-profile runs (`<name>--test`) only prove the pipeline: they are left out below
unless `INCLUDE_TESTS` is set.

In [ ]:
INCLUDE_TESTS = True  # only test runs exist so far; set False once real ones do

runs = results.load_runs(results.find_runs())
if not INCLUDE_TESTS:
    runs = [r for r in runs if not r.settings["test"]]
table = results.overview(runs)
print(results.sources())
table

Pick the runs to compare: an experiment (and target), or any list of keys.

In [ ]:
EXPERIMENT = table.experiment.iloc[0] if len(table) else None  # e.g. "market_3x3"
TARGET = None  # e.g. "taltech"; None: every target

chosen = [
    r
    for r in runs
    if r.ref.experiment == EXPERIMENT and (TARGET is None or r.ref.target == TARGET)
]
print(len(chosen), "runs:", [r.key for r in chosen])
print("settings that differ:", results.differing_settings([r.settings for r in chosen]))

## 1. Did it learn?

The trained policy's evaluation return during training (the evaluation episodes, the
same seeds at every point), with the baselines at step 0 as a reference. Training
return (what the sampled episodes earned while learning) is a different measurement:
`results.training_curves`.

In [ ]:
curves = results.learning_curves(chosen)
fig, ax = plt.subplots(figsize=(9, 4))
for key, run in curves[curves.policy == "trained"].groupby("key"):
    ax.plot(run.env_steps, run.team_return_mean, marker="o", label=key.split("/")[-1])
start = curves[(curves.env_steps == 0) & (curves.policy != "trained")]
for policy, rows in start.groupby("policy"):
    ax.axhline(rows.team_return_mean.mean(), ls="--", lw=1, color="grey")
    ax.annotate(policy, (0, rows.team_return_mean.mean()), fontsize=8, color="grey")
ax.set_xlabel("environment steps")
ax.set_ylabel("team return (evaluation)")
ax.legend(fontsize=8, ncol=3)
plt.show()

## 2. How good is the final policy?

Each run on every test day once: trained against each baseline on the **same** day
(`trained_minus_<baseline>` > 0: the trained policy did better). A run from before the
per-day tables gives one row with its means (`day` empty).

In [ ]:
days = results.heldout_days(chosen)
days.groupby(["key"])[["trained", "zero", "net_position", "trained_minus_zero", "trained_minus_net_position"]].mean()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
labels = []
for i, (key, run) in enumerate(days.groupby("key")):
    labels.append(key.split("/")[-1])
    for offset, column in ((-0.15, "trained_minus_zero"), (0.15, "trained_minus_net_position")):
        ax.scatter([i + offset] * len(run), run[column], s=12)
        ax.plot([i + offset - 0.1, i + offset + 0.1], [run[column].mean()] * 2, color="black")
ax.axhline(0, color="grey", lw=1)
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_ylabel("trained − baseline, per test day")
ax.set_title("left of each run: − zero, right: − net_position (bar: mean)")
plt.show()

Per participant, one run: who gains on which day. A better team total can hide
participants that lose.

In [ ]:
if chosen:
    per = results.participant_days(chosen[0])
    if per.empty:
        print("no per-participant table (a run from before it was written)")
    else:
        grid = per.pivot(index="agent", columns="day", values="trained_minus_net_position")
        fig, ax = plt.subplots(figsize=(9, 0.4 * len(grid) + 1.5))
        image = ax.imshow(grid.values, cmap="RdBu", aspect="auto")
        ax.set_yticks(range(len(grid)), grid.index)
        ax.set_xticks(range(len(grid.columns)), grid.columns, rotation=45, ha="right")
        fig.colorbar(image, label="trained − net_position (return)")
        ax.set_title(chosen[0].key)
        plt.show()

## 3. How consistent is it?

Each configuration (experiment, algorithm, paradigm, variant) over its seeds: the mean
of the seeds' held-out returns, their spread (`std` over seeds: 0 with one seed), the
number of completed seeds, and `differs`: settings that are not the same in its runs
(scenario, test days, budget, commit, results detail), which would make the group not
one configuration.

In [ ]:
results.seed_groups(chosen)